## Cell 1 — Load model & artifacts


In [ ]:
# ============================================================
# CELL 1 — Load toàn bộ artifacts cần thiết để predict
# ============================================================
import json, re, torch
import torch.nn as nn
import numpy as np

# ── Đường dẫn — chỉnh lại nếu cần ──────────────────────────────────────
CHECKPOINT  = r'C:\Users\Admin\Documents\nlp\NLP_Project\output\models\checkpoints\best_model.pt'
VOCAB_FILE  = r'C:\Users\Admin\Documents\nlp\NLP_Project\data\process_data\vocab.json'
LABEL_FILE  = r'C:\Users\Admin\Documents\nlp\NLP_Project\data\process_data\label_map.json'
STOPWORDS_FILE = r'C:\Users\Admin\Documents\nlp\NLP_Project\data\dictionary\vietnamese-stopwords.txt'

# ── Load files ───────────────────────────────────────────────────────────
with open(VOCAB_FILE,  encoding='utf-8') as f: vocab     = json.load(f)
with open(LABEL_FILE,  encoding='utf-8') as f: label_map = json.load(f)
with open(STOPWORDS_FILE, encoding='utf-8') as f:
    STOPWORDS = {line.strip() for line in f if line.strip()}

# idx → tên nhãn (để hiển thị kết quả)
IDX_TO_LABEL = {
    level: {v: k for k, v in label_map[level].items()}
    for level in ['l1', 'l2', 'l3']
}
NUM_CLASSES = [len(label_map['l1']), len(label_map['l2']), len(label_map['l3'])]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Vocab  : {len(vocab):,} từ')
print(f'Nhãn   : L1={NUM_CLASSES[0]}  L2={NUM_CLASSES[1]}  L3={NUM_CLASSES[2]}')

# ── HARNN phải khớp tên layer với lúc train (emb, bigru, drop, ...) ─────
class HARNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size,
                 num_classes_per_level, dropout=0.5):
        super().__init__()
        self.num_levels = len(num_classes_per_level)
        self.hidden_size = hidden_size

        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bigru = nn.GRU(embed_dim, hidden_size, bidirectional=True, batch_first=True)
        self.drop = nn.Dropout(dropout)

        self.attention = nn.ModuleList([
            nn.Linear(hidden_size * 2, 1) for _ in range(self.num_levels)
        ])
        self.ham = nn.LSTMCell(hidden_size * 2, hidden_size)
        self.classifiers = nn.ModuleList([
            nn.Linear(hidden_size * 3, n) for n in num_classes_per_level
        ])

    def forward(self, x):
        doc = self.drop(self.bigru(self.drop(self.emb(x)))[0])
        h = torch.zeros(x.size(0), self.hidden_size, device=x.device)
        c = torch.zeros(x.size(0), self.hidden_size, device=x.device)

        preds = []
        for lv in range(self.num_levels):
            context = (torch.softmax(self.attention[lv](doc), dim=1) * doc).sum(dim=1)
            h, c = self.ham(context, (h, c))
            feat = self.drop(torch.cat([context, h], dim=-1))
            preds.append(self.classifiers[lv](feat))  # logits
        return preds

# ── Load checkpoint ───────────────────────────────────────────────────────
EMBED_DIM   = 100
HIDDEN_SIZE = 256

model = HARNN(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_classes_per_level=NUM_CLASSES,
).to(device)

ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'\nLoaded checkpoint epoch {ckpt["epoch"]}'
      f' (val F1-L1={ckpt["val_f1_l1"]:.3f})')


## Cell 2 — Hàm predict
Nhận vào text, trả về nhãn kèm xác suất cho từng level.


In [ ]:
# ============================================================
# CELL 2 — Hàm predict
# ============================================================
MAX_LEN   = 512
THRESHOLD = 0.5   # Hạ xuống 0.3 nếu muốn recall cao hơn

def clean_text(text: str) -> str:
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def tokenize(text: str) -> list[str]:
    try:
        from underthesea import word_tokenize
        tokens = word_tokenize(clean_text(text), format='text').split()
    except ImportError:
        tokens = clean_text(text).split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def text_to_tensor(text: str) -> torch.Tensor:
    tokens = tokenize(text)
    ids    = [vocab.get(t, 1) for t in tokens][:MAX_LEN]
    ids   += [0] * (MAX_LEN - len(ids))
    return torch.tensor([ids], dtype=torch.long).to(device)   # (1, MAX_LEN)

@torch.no_grad()
def predict(
    text: str,
    threshold: float = THRESHOLD,
    top_k: int = 3,       # nếu không có nhãn nào qua threshold → lấy top_k
) -> dict:
    """
    Dự đoán nhãn cho một đoạn văn bản.

    Trả về dict:
    {
        'l1': [{'label': 'Thể thao', 'prob': 0.97}],
        'l2': [{'label': 'Bóng đá',  'prob': 0.89}],
        'l3': [],
        'tokens': ['mbappe', 'ghi_bàn', ...]
    }
    """
    x      = text_to_tensor(text)
    logits = model(x)          # list of (1, num_classes_i)
    tokens = tokenize(text)

    result = {'tokens': tokens[:10]}

    for i, level in enumerate(['l1', 'l2', 'l3']):
        probs     = torch.sigmoid(logits[i][0]).cpu().numpy()  # (num_classes,)
        idx2label = IDX_TO_LABEL[level]

        # Lấy nhãn vượt threshold
        selected = [
            {'label': idx2label[j], 'prob': float(probs[j])}
            for j in range(len(probs))
            if probs[j] >= threshold
        ]

        # Nếu không có nhãn nào → lấy top_k nhãn cao nhất
        if not selected:
            top_indices = np.argsort(probs)[::-1][:top_k]
            selected    = [
                {'label': idx2label[j], 'prob': float(probs[j])}
                for j in top_indices
            ]

        # Sắp xếp theo prob giảm dần
        selected.sort(key=lambda x: x['prob'], reverse=True)
        result[level] = selected

    return result


def print_result(text: str, result: dict) -> None:
    """In kết quả đẹp ra màn hình."""
    print('─' * 60)
    print(f'Văn bản : {text[:80]}...' if len(text) > 80 else f'Văn bản : {text}')
    print(f'Tokens  : {result["tokens"]} ...')
    print()
    for level in ['l1', 'l2', 'l3']:
        labels = result[level]
        print(f'  {level.upper()}:')
        for item in labels:
            bar   = '█' * int(item['prob'] * 20)
            mark  = '✓' if item['prob'] >= THRESHOLD else '·'
            print(f'    {mark} {item["label"]:<25} {item["prob"]:>5.1%}  {bar}')
    print()

print('Hàm predict sẵn sàng!')
print(f'Threshold: {THRESHOLD}  (hạ xuống 0.3 nếu muốn recall cao hơn)')


## Cell 3 — Predict bài viết của bạn
Nhập text trực tiếp để dự đoán.


In [ ]:
# ============================================================
# CELL 4A — Nhập text trực tiếp
# Thay YOUR_TEXT bằng bài viết muốn dự đoán
# ============================================================

YOUR_TEXT = """
(Dân trí) - Google - công ty con của tập đoàn Alphabet - vừa công bố khoản chi trị giá 1 tỷ USD sử dụng trong vòng 3 năm nhằm hỗ trợ đào tạo về trí tuệ nhân tạo (AI) tại các trường đại học trên khắp nước Mỹ.
Tính đến nay, hơn 100 trường đại học của Mỹ đã đăng ký tham gia kế hoạch này của Google, bao gồm một số hệ thống trường đại học công lập lớn nhất nước Mỹ như Texas A&M hay Đại học Bắc Carolina.

Khoản hỗ trợ sẽ bao gồm cả tiền mặt và các nguồn lực như tín dụng điện toán đám mây phục vụ trong quá trình đào tạo sinh viên, cũng như hỗ trợ các nghiên cứu liên quan đến AI tại các trường đại học.

Google cam kết chi 1 tỷ USD để đào tạo AI tại các trường đại học Mỹ - 1
Google sẽ chi 1 tỷ USD để đào tạo AI tại các trường đại học của Mỹ (Ảnh minh họa: CNA).

Đáng chú ý, gói hỗ trợ còn tính gộp cả giá trị của các công cụ AI đòi hỏi trả phí mà Google đang cung cấp trên thị trường. Khi tham gia vào công tác đào tạo AI tại các trường đại học, Google sẽ cung cấp miễn phí các công cụ này cho giảng viên và sinh viên tại những trường hợp tác đào tạo với Google.

Ông James Manyika - Phó Chủ tịch của Google - cho biết công ty này đang đặt mục tiêu mở rộng chương trình đến toàn bộ các trường đại học phi lợi nhuận được công nhận tại Mỹ. Đồng thời, Google cũng đang lên kế hoạch triển khai kế hoạch tương tự ở một số quốc gia khác.

TIN LIÊN QUAN
AI của Google và OpenAI "đoạt huy chương vàng Olympic toán quốc tế"
Thông báo này được đưa ra trong bối cảnh các đối thủ của Google như OpenAI, Anthropic và Amazon cũng đang thúc đẩy các sáng kiến liên quan đến AI trong lĩnh vực giáo dục, khi AI ngày càng thâm nhập sâu vào đời sống xã hội. Trong tháng 7, Microsoft cũng đã đưa ra cam kết chi 4 tỷ USD để thúc đẩy ứng dụng AI trong giáo dục trên toàn cầu.

Thông qua việc phổ biến sản phẩm AI tới sinh viên, các hãng công nghệ kỳ vọng sẽ giành lợi thế trong các thương vụ kinh doanh của tập đoàn mình trong tương lai, khi nhóm người dùng trẻ tuổi này thực sự bước vào thị trường lao động.

Hiện nay, nhiều nghiên cứu đã chỉ ra mối lo ngại xung quanh AI trong lĩnh vực giáo dục, từ việc tạo điều kiện cho gian lận trong nghiên cứu và thi cử, cho đến nguy cơ làm suy giảm năng lực tư duy phản biện của người dùng. Một số trường thậm chí đã áp dụng lệnh cấm sinh viên sử dụng AI trong quá trình viết luận và thực hiện nghiên cứu.

Tuy nhiên, ông Manyika cho biết kể từ khi Google bắt đầu xây dựng sáng kiến giáo dục này hồi đầu năm nay, công ty chưa gặp phải sự phản đối nào từ phía ban giám hiệu các trường đại học. Dù vậy, ông Manyika cũng thừa nhận rằng vẫn còn “rất nhiều câu hỏi” cần được làm rõ xung quanh AI.

“Chúng tôi hy vọng có thể cùng các trường đại học tìm hiểu cách sử dụng tốt nhất các công cụ AI. Những hiểu biết thu được sẽ góp phần giúp định hình các quyết định của chúng tôi trong việc xây dựng sản phẩm trong tương lai”, ông Manyika nói thêm.
"""

if YOUR_TEXT.strip() and 'Paste' not in YOUR_TEXT:
    result = predict(YOUR_TEXT)
    print_result(YOUR_TEXT, result)
else:
    print('Thay YOUR_TEXT bằng nội dung bài viết của bạn')
